# 01 — Data Exploration

Explore the concrete crack dataset:
- Image count and file extension breakdown
- Image size distribution
- Crack density (positive pixel ratio) per image
- Sample image + mask visualisation

In [ ]:
import sys
sys.path.insert(0, '..')

import numpy as np
import matplotlib.pyplot as plt
from pathlib import Path
from PIL import Image

DATASET_ROOT = Path('../concreteCrackSegmentationDataset')
rgb_dir = DATASET_ROOT / 'rgb'
bw_dir  = DATASET_ROOT / 'BW'

In [ ]:
# Count images and extensions
rgb_paths = sorted(rgb_dir.iterdir())
exts = {}
for p in rgb_paths:
    exts[p.suffix] = exts.get(p.suffix, 0) + 1
print(f'Total rgb images: {len(rgb_paths)}')
print(f'Extensions: {exts}')

In [ ]:
# Image size distribution
sizes = [Image.open(p).size for p in rgb_paths if p.suffix.lower() in ('.jpg', '.jpeg', '.png')]
widths  = [s[0] for s in sizes]
heights = [s[1] for s in sizes]
print(f'Width  — min: {min(widths)}, max: {max(widths)}, mean: {np.mean(widths):.0f}')
print(f'Height — min: {min(heights)}, max: {max(heights)}, mean: {np.mean(heights):.0f}')

In [ ]:
# Crack density (fraction of white pixels in BW masks)
densities = []
for p in sorted(bw_dir.iterdir()):
    if p.suffix.lower() not in ('.jpg', '.jpeg', '.png'):
        continue
    arr = np.array(Image.open(p).convert('L')) > 127
    densities.append(arr.mean())

plt.figure(figsize=(8, 4))
plt.hist(densities, bins=40, edgecolor='black')
plt.xlabel('Crack pixel density')
plt.ylabel('Image count')
plt.title('Distribution of crack pixel density across all images')
plt.tight_layout()
plt.show()
print(f'Mean density: {np.mean(densities):.4f}  Median: {np.median(densities):.4f}')

In [ ]:
# Visualise 4 sample pairs
rgb_index = {p.stem.lower(): p for p in rgb_paths if p.suffix.lower() in ('.jpg','.jpeg','.png')}
bw_paths  = sorted([p for p in bw_dir.iterdir() if p.suffix.lower() in ('.jpg','.jpeg','.png')])

fig, axes = plt.subplots(4, 2, figsize=(10, 16))
for i, bw_p in enumerate(bw_paths[:4]):
    rgb_p = rgb_index.get(bw_p.stem.lower())
    axes[i, 0].imshow(Image.open(rgb_p))
    axes[i, 0].set_title(f'RGB — {bw_p.stem}')
    axes[i, 1].imshow(Image.open(bw_p), cmap='gray')
    axes[i, 1].set_title('BW Mask')
for ax in axes.ravel():
    ax.axis('off')
plt.tight_layout()
plt.show()